INIT

In [39]:
from pathlib import Path
from typing import Final

import pandas as pd


STAGING_DATA_PATH: Final[Path] = Path("..") / "data" / "staging"
PROCESSED_DATA_PATH: Final[Path] = STAGING_DATA_PATH.parent / "dwh"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

EXTRACT

In [40]:
customers_df: pd.DataFrame = pd.read_csv(STAGING_DATA_PATH / "customers.csv")
customers_df.info()
customers_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        200 non-null    int64 
 1   region             200 non-null    object
 2   registration_date  200 non-null    object
dtypes: int64(1), object(2)
memory usage: 4.8+ KB


,customer_id,region,registration_date
0,101,West,2023-01-01
1,102,Central,2023-01-02
2,103,East,2023-01-03
3,104,West,2023-01-04
4,105,South,2023-01-05


In [41]:
products_df: pd.DataFrame = pd.read_csv(STAGING_DATA_PATH / "products.csv")
products_df.info()
products_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  20 non-null     int64  
 1   category    20 non-null     object 
 2   price       20 non-null     float64
 3   cost_price  20 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 772.0+ bytes


,product_id,category,price,cost_price
0,1,Home,337.85,186.37
1,2,Home,280.83,107.28
2,3,Clothing,233.52,130.05
3,4,Home,179.65,172.49
4,5,Electronics,343.85,48.44


In [42]:
orders_df: pd.DataFrame = pd.read_csv(STAGING_DATA_PATH / "orders.csv")
orders_df.info()
orders_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   order_id     2000 non-null   int64 
 1   order_date   2000 non-null   object
 2   customer_id  2000 non-null   int64 
 3   product_id   2000 non-null   int64 
 4   quantity     2000 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 78.3+ KB


,order_id,order_date,customer_id,product_id,quantity
0,1001,2024-11-07,181,19,2
1,1002,2024-01-21,297,15,2
2,1003,2024-06-29,130,1,4
3,1004,2024-06-11,197,11,2
4,1005,2024-03-30,198,16,1


TRANSFORM

In [43]:
# Convert date objects to datetime

customers_df['registration_date'] = pd.to_datetime(customers_df['registration_date'], errors='raise')
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'], errors='raise')
print("Conversion [registration_date, order_date] to datetime is successful\n")

Conversion [registration_date, order_date] to datetime is successful



In [44]:
# Fix duplicated

customers_df = customers_df.drop_duplicates(subset=['customer_id'], keep='first')
products_df = products_df.drop_duplicates(subset=['product_id'], keep='first')
orders_df = orders_df.drop_duplicates()

print("Customers duplicated:", customers_df.duplicated().sum())
print("Products duplicated:", products_df.duplicated().sum())
print("Orders duplicated:", orders_df.duplicated().sum())

Customers duplicated: 0
Products duplicated: 0
Orders duplicated: 0


In [45]:
# Checking the correctness of prices and quantities

products_df = products_df.query("price > 0 and cost_price > 0")
orders_df = orders_df.query("quantity > 0")


In [46]:
# Update index

customers_df = customers_df.reset_index(drop=True)
products_df = products_df.reset_index(drop=True)
orders_df = orders_df.reset_index(drop=True)

LOAD

In [47]:
# Save dim and fact tables 

print("Rows before merged:", len(orders_df))
fact_order_df: pd.DataFrame = orders_df.merge(products_df, on="product_id", how="inner")
fact_order_df = fact_order_df.merge(customers_df, on="customer_id", how="inner")
print("Rows after merged:", len(fact_order_df))

customers_df.to_csv(PROCESSED_DATA_PATH / "dim_customers.csv", index=False)
products_df.to_csv(PROCESSED_DATA_PATH / "dim_products.csv", index=False)
orders_df.to_csv(PROCESSED_DATA_PATH / "dim_orders.csv", index=False)
fact_order_df.to_csv(PROCESSED_DATA_PATH / "fact_order.csv", index=False)
print("\nNormalized data has been successfully saved")

Rows before merged: 2000
Rows after merged: 2000

Normalized data has been successfully saved
